# DESCRIPTION (inspired by Leetcode.com)
Write a function to find the length of the longest substring containing the same letter in a given string s, after performing at most k operations in which you can choose any character of the string and change it to any other uppercase English letter.

Input:

s = "BBABCCDD"
k = 2

Output:

5

Explanation: Replace the first 'A' and 'C' with 'B' to form "BBBBBCDD". The longest substring with identical letters is "BBBBB", which has a length of 5.

In [ ]:
class Solution_V1:
    def characterReplacement(self, s: str, k: int) -> int:
        if k >= len(s)-1:
            return len(s)
        
        start, max_len = 0, 0
        freq_map = {} # keep track of letter occurence within the substring (window)

        for end in range(len(s)):
            freq_map[s[end]] = freq_map.get(s[end],0) + 1
            
            if end - start + 1 > freq_map[s[end]] + k:
                # slide window
                freq_map[s[start]] -= 1
                start += 1
            else:
                # window is expanded, update max_len as needed
                max_len = max(max_len, freq_map[s[end]] + k)

        return max_len 

# FEEDBACK

Your sliding-window idea is right, but the window validity check and max update are incorrect.

Main bug:

max_len = max(max_len, freq_map[s[end]] + k) is not the substring length. It can exceed len(s), which is why "AAAA", 2 returns 6.
You’re also only shrinking once with if, but a window may need to shrink repeatedly.
What to use instead:

Track the most frequent character count in the current window.
The window is valid when window_size - max_freq <= k.
Update answer with the actual window length.
Also, the early return if k >= len(s)-1 is unnecessary and can be misleading; the general sliding-window logic already handles it.

In [ ]:
class Solution_V2:
    def characterReplacement(self, s: str, k: int) -> int:
        # max_freq keeps track of the highest occurence for any letter within the window
        # max_len is the largest window (subarray) that meets the criteria (window size <= max_freq + k)
        start, max_freq, max_len = 0, 0, 0 
        freq_map = {} # keep track of letter occurence within the substring (window)

        for end in range(len(s)):
            freq_map[s[end]] = freq_map.get(s[end],0) + 1
            max_freq = max(max_freq, freq_map[s[end]])

            if end-start+1 <= max_freq + k:
                max_len = max(max_len, end-start+1)
            else:
                # slide window
                freq_map[s[start]] -= 1
                start += 1
                
        return max_len 

# FEEDBACK 2
Good job — your solution is correct and the sliding-window idea is exactly what this problem needs.

A couple of interview-level notes:

Your validity check is right in spirit, but you only shrink the window by one character when invalid. Usually you want to keep shrinking until the window becomes valid again (while, not if), otherwise the window can stay invalid for several iterations.
Because max_freq is monotonic, your approach still passes, but it’s worth understanding why this works: max_freq may become stale, yet it never causes an incorrect answer in this pattern.

You can update max_len after the window is valid, which makes the logic easier to reason about.
Overall: solid solution with the right complexity, O(n) time and O(1) extra space for the alphabet.


In [ ]:
class Solution:
    def characterReplacement(self, s: str, k: int) -> int:
        # max_freq keeps track of the highest occurence for any letter that has appeared in any window so far.
        # max_len is the largest window (subarray) that meets the criteria (window size <= max_freq + k)
        start, max_freq, max_len = 0, 0, 0 
        freq_map = {} # keep track of letter occurence within the substring (window)

        for end in range(len(s)):
            freq_map[s[end]] = freq_map.get(s[end],0) + 1
            max_freq = max(max_freq, freq_map[s[end]])

            while end-start+1 > max_freq + k:
                # slide window until meeting criteria
                freq_map[s[start]] -= 1
                start += 1

            max_len = max(max_len, end-start+1)
                
                
        return max_len 

# FEEDBACK 3

Correct solution — nice sliding window.

A couple of interview-level notes:

Your invariant is right: a window is valid when window_size - max_freq <= k.
Since max_freq is only ever increased, it can become stale after shrinking. That’s okay here; it won’t break correctness, but it’s worth being able to explain why the window still works.
Small cleanup: your comment says window size <= max_freq + k, which is equivalent, but the more common form is window_size - max_freq <= k.
You handled edge cases implicitly well, including s == "" and k >= len(s).



In [1]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Input:
    s: str
    k: int
    
@dataclass(frozen=True)
class Test:
    input: Input
    expected_result: int

def run_tests(tests: list[Test], func: Callable[[str, int], int]):
    for test in tests:
        result = func(test.input.s, test.input.k)
        if result == test.expected_result:
            print("Test passed for " + str(test.input.s))
        else:
            print(f"Test failed for {test.input.s}. Expected: {test.expected_result}, Actual: {result}")

In [45]:
tests = [
    Test(Input("BBABCCDD",2),5),
    Test(Input("ABC",3),3),
    Test(Input("ABC",2),3),
    Test(Input("ABC",1),2),
    Test(Input("abb",1),3),
    Test(Input("bbc",1),3),
    Test(Input("acc",0),2),
    Test(Input("acca",1),3),
    Test(Input("acca",2),4),
    Test(Input("aacc",3),4),
    Test(Input("ccaa",5),4),
    Test(Input("aaaa",2),4)
]

run_tests(tests, Solution().characterReplacement)

Test passed for BBABCCDD
Test passed for ABC
Test passed for ABC
Test passed for ABC
Test passed for abb
Test passed for bbc
Test passed for acc
Test passed for acca
Test passed for acca
Test passed for aacc
Test passed for ccaa
Test passed for aaaa
